In [1]:
#import neessary libraries for feature engineering
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix,hstack
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import OneHotEncoder

In [2]:
#load the processed datasets
training = pd.read_csv(r'C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed\training_data.csv')
test = pd.read_csv(r'C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed\test_data.csv')

In [3]:
training.head()

,unique_hash,text,drug,sentiment,cleaned_text,clean_drug
0,2e180be4c9214c1f5ab51fd8cc32bc80c9f612e0,Autoimmune diseases tend to come in clusters. ...,gilenya,2,autoimmune disease tend come cluster gilenya f...,gilenya
1,9eba8f80e7e20f3a2f48685530748fbfa95943e4,I can completely understand why you’d want to ...,gilenya,2,completely understand youd want try result rep...,gilenya
2,fe809672251f6bd0d986e00380f48d047c7e7b76,Interesting that it only targets S1P-1/5 recep...,fingolimod,2,interesting target sp receptor rather like fin...,fingolimod
3,bd22104dfa9ec80db4099523e03fae7a52735eb6,"Very interesting, grand merci. Now I wonder wh...",ocrevus,2,interesting grand merci wonder lemtrada ocrevu...,ocrevus
4,b227688381f9b25e5b65109dd00f7f895e838249,"Hi everybody, My latest MRI results for Brain ...",gilenya,1,hi everybody latest mri result brain cervical ...,gilenya


In [4]:
# check number of unique words in the cleaned_text column
unique_words = set(" ".join(training['cleaned_text']).split())

print("Vocabulary size:", len(unique_words))

Vocabulary size: 40035


In [5]:
from collections import Counter

all_words = " ".join(training['cleaned_text']).split()

word_counts = Counter(all_words)

rare_words = [word for word, count in word_counts.items() if count == 1]

print("Rare words:", len(rare_words))

Rare words: 17396


# Observation

The dataset contains 40,035 unique words, out of which 17,396 words appear only once across all reviews. This indicates a high proportion of rare tokens, which is common in natural language datasets due to spelling variations, domain-specific terminology, and noise.

To reduce dimensionality and improve model generalization, rare words will be filtered during TF-IDF feature extraction using a minimum document frequency threshold (min_df).

Some drugs appeared very infrequently in the dataset, with certain drugs occurring only once. Such rare categories can introduce noise when using one-hot encoding because the model cannot learn reliable patterns from a single observation. To mitigate this, drugs appearing fewer than a specified threshold were grouped into an other_drug category before encoding.

In [6]:
# vectorizer
tfidf = TfidfVectorizer(
    max_features=6000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

In [7]:
# checking rare drugs 
drug_counts = training['clean_drug'].value_counts()

rare_drugs = drug_counts[drug_counts < 5].index

In [8]:
rare_drugs

Index(['aubagio', 'photodynamic therapy', 'cyltezo', 'necitumumab', 'mekinist',
       'dexamethasone implant', 'remsima', 'certolizumab pegol', 'aflibercept',
       'macugen', 'arzerra', 'zykadia', 'tafinlar', 'ceritinib', 'ixifi',
       'brolucizumab', 'amjevita', 'teriflunomide', 'trametinib', 'lorlatinib',
       'dabrafenib', 'pan-retinal photocoagulation', 'infliximab-dyyb',
       'geftinib', 'pf-00547659', 'alunbrig', 'pemetrexed disodium',
       'portrazza', 'movectro', 'brigatinib', 'pegaptanib', 'alemtuzumab',
       'rhumab 2h7', 'filgotinib', 'alectnib', 'ct-p13', 'guselkumab',
       'cyramza'],
      dtype='object', name='clean_drug')

In [9]:
# repalcing rare drugs with "other drugs"
training['clean_drug'] = training['clean_drug'].replace(rare_drugs, 'other_drug')
test['clean_drug'] = test['clean_drug'].replace(rare_drugs, 'other_drug')

In [10]:
#get only the needed columns for feature engineering
train = training[["cleaned_text", "clean_drug", "sentiment"]]
testing = test[["cleaned_text", "clean_drug"]]

In [11]:
# Drug mention feature
train['drug_in_text'] = train.apply(
    lambda x: 1 if x['clean_drug'] in x['cleaned_text'] else 0,
    axis=1

)

testing['drug_in_text'] = testing.apply(
    lambda x: 1 if x['clean_drug'] in x['cleaned_text'] else 0,
    axis=1
)

C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_32428\1484315527.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['drug_in_text'] = train.apply(
C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_32428\1484315527.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  testing['drug_in_text'] = testing.apply(


In [ ]:
# defining list of positive and negative words for feature enginnering
positive_words = [
    "worked","effective","great","excellent","relief",
    "helped","improved","better","amazing","perfect", "improvement"
    "recommended","success","good","love","progress","breakthrough",
    "promising", "best", "hope", "positive", "fantastic", "benefit"
]

negative_words = [
    "pain","nausea","vomit","vomiting","dizziness",
    "fatigue","rash","headache","diarrhea","cramps",
    "terrible","awful","worse","bad","horrible", "failure",
    "miserable", "fail", "horrifying", "infection", "terrified" 'terrible',
    "coughing", "insane"
]

In [13]:
train[train['sentiment'] == 1]['cleaned_text'].iloc[10]

'daughter tested ro positive nsclc taking crizotinib week recent scan ct bone show slight progression liver possible new spot bone spine rib cancer lung node appears stable even smaller blood work show high ldh low albumin reason believe crizotinib effective possible ro detection false positive testing performed paraffin embedded biopsy tissue fluorescence situ hybridization cell positive ro gene rearrangement topic modified year month ago fanos topic modified year month ago fanos topic modified year month ago fanos topic modified year month ago fanos'

In [ ]:
# function to count positive words
def positive_score(text):
    words = text.split()
    return sum(word in positive_words for word in words)

train['positive_score'] = train['cleaned_text'].apply(positive_score)
testing['positive_score'] = testing['cleaned_text'].apply(positive_score)

C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_32428\3954458097.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['positive_score'] = train['cleaned_text'].apply(positive_score)
C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_32428\3954458097.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  testing['positive_score'] = testing['cleaned_text'].apply(positive_score)


In [ ]:
#function to count negative words
def negative_score(text):
    words = text.split()
    return sum(word in negative_words for word in words)

train['negative_score'] = train['cleaned_text'].apply(negative_score)
testing['negative_score'] = testing['cleaned_text'].apply(negative_score)

In [ ]:
# create sentiment strength features
train['sentiment_strength'] = train['positive_score'] - train['negative_score']
testing['sentiment_strength'] = testing['positive_score'] - testing['negative_score']

In [ ]:
#list of side effects
side_effect_words = [
    "side effect","caused","reaction","bleeding",
    "nausea","dizziness","headache", "fatigue", "diarrhea", "abdominal pain", "vomiting", "constipation", "dry mouth", "insomnia", 
    "rash", "itching", "swelling", "difficulty breathing", "chest pain", "irregular heartbeat", "seizures", 
    "muscle pain", "joint pain", "blurred vision", "hearing loss", "anxiety", "depression"
]


In [ ]:
# function to check for side effects
def side_effect_flag(text):
    return int(any(word in text for word in side_effect_words))

train['side_effect_flag'] = train['cleaned_text'].apply(side_effect_flag)
testing['side_effect_flag'] = testing['cleaned_text'].apply(side_effect_flag)

In [ ]:
#list of words indicating drug effectiveness
effect_words = [
    "worked","relief","effective","improved","helped"
]

In [ ]:
# fuc t0 check for drug effectiveness
def effectiveness_flag(text):
    return int(any(word in text for word in effect_words))

train['effect_flag'] = train['cleaned_text'].apply(effectiveness_flag)
testing['effect_flag'] = testing['cleaned_text'].apply(effectiveness_flag)

In [ ]:
#list of neutral words
neutral_words = [
    "started","taking","prescribed","week","month","day"
]

def neutral_score(text):
    words = text.split()
    return sum(word in neutral_words for word in words)

train['neutral_score'] = train['cleaned_text'].apply(neutral_score)
testing['neutral_score'] = testing['cleaned_text'].apply(neutral_score)

In [ ]:
#create pos to neg ratio feature
train['pos_neg_ratio'] = train['positive_score'] / (train['negative_score'] + 1)
testing['pos_neg_ratio'] = testing['positive_score'] / (testing['negative_score'] + 1)

In [ ]:
#checking the engineered features
train.head()

,cleaned_text,clean_drug,sentiment,drug_in_text,positive_score,negative_score,sentiment_strength,side_effect_flag,effect_flag,neutral_score,pos_neg_ratio
0,autoimmune disease tend come cluster gilenya f...,gilenya,2,1,2,0,2,0,0,2,2.0
1,completely understand youd want try result rep...,gilenya,2,1,3,0,3,0,1,0,3.0
2,interesting target sp receptor rather like fin...,fingolimod,2,1,0,0,0,0,0,0,0.0
3,interesting grand merci wonder lemtrada ocrevu...,ocrevus,2,1,0,0,0,0,0,0,0.0
4,hi everybody latest mri result brain cervical ...,gilenya,1,1,1,0,1,0,0,1,1.0


In [24]:
train.columns

Index(['cleaned_text', 'clean_drug', 'sentiment', 'drug_in_text',
       'positive_score', 'negative_score', 'sentiment_strength',
       'side_effect_flag', 'effect_flag', 'neutral_score', 'pos_neg_ratio'],
      dtype='object')

In [ ]:
#one hot encoding foor drugs
one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)


# define feature and target variables
X_text = train['cleaned_text']
X_drug = one_hot_encoder.fit_transform(train[["clean_drug"]])
y = train['sentiment']

In [33]:
#vectorize the text data
X_tfidf = tfidf.fit_transform(X_text)

In [36]:
# convert one-hot encoding to sparese marix

X_drug_sparse = csr_matrix(X_drug)

In [ ]:
# combine the text and drug features

X = hstack([
    X_tfidf,
    X_drug_sparse,
    csr_matrix(train[['drug_in_text',
                      'positive_score',
                      'negative_score',
                      'sentiment_strength',
                      'side_effect_flag',
                      'effect_flag',
                      'neutral_score',
                      'pos_neg_ratio']].values)
])

In [38]:
X.shape

(5279, 6068)

In [39]:
y.shape

(5279,)

In [ ]:
# over sampling the minority class
ros = RandomOverSampler(
    sampling_strategy={0:1200, 1:1200}
)

X_over, y_over = ros.fit_resample(X, y)

In [ ]:
# undersampling the majority class
rus = RandomUnderSampler(
    sampling_strategy={2:2000}
)

X_resampled, y_resampled = rus.fit_resample(X_over, y_over)

In [42]:
X_resampled.shape

(4400, 6068)

In [43]:
X.shape

(5279, 6068)

In [44]:
#spliting the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.2,
    random_state=42,
    stratify=y_resampled
)

In [45]:
#feature engineering for the test dataset
testing_text = testing['cleaned_text']
testing_drug = one_hot_encoder.transform(testing[["clean_drug"]])


testing_tfidf = tfidf.transform(testing_text)
#testing_drug_encoded = one_hot_encoder.transform(testing_drug)
testing_drug_sparse = csr_matrix(testing_drug)
final_testing = hstack([testing_tfidf, testing_drug_sparse,csr_matrix(testing[['drug_in_text',
                      'positive_score',
                      'negative_score',
                      'sentiment_strength',
                      'side_effect_flag',
                      'effect_flag',
                      'neutral_score',
                      'pos_neg_ratio']].values)])

In [46]:
final_testing.shape

(2924, 6068)

In [47]:
X.shape

(5279, 6068)

In [49]:
import joblib

# Save training features and target
joblib.dump(X_train, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/X_train.pkl")
joblib.dump(X_val, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/X_val.pkl")
joblib.dump(y_train, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/y_train.pkl")
joblib.dump(y_val, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/y_val.pkl")

# Save test features
joblib.dump(final_testing, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/X_test.pkl")
joblib.dump(tfidf, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/tfidf_vectorizer.pkl")
#joblib.dump(X_drug_encoded.columns, r"C:\Users\GIGABYTE\Desktop\drug-sentiment-analysis\data\processed/drug_columns.pkl")

['C:\\Users\\GIGABYTE\\Desktop\\drug-sentiment-analysis\\data\\processed/tfidf_vectorizer.pkl']